# Notebook 16 — Dive-Computer Estimation Pipeline

**Companion to Chapter 16**

This laboratory separates sensor measurements, derived variables, filtered estimates, integrated states, and alarms.

> It is not certified or suitable for real dive-computer use.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True})
rho, g, p0 = 1025.0, 9.80665, 101325.0

def ambient_pressure_pa(depth_m):
    return p0 + rho*g*np.asarray(depth_m)

def ambient_pressure_bar(depth_m):
    return ambient_pressure_pa(depth_m)/1e5

## 1. Synthetic depth and pressure measurements

In [ ]:
def true_depth(t):
    return np.piecewise(t,[t<180,(t>=180)&(t<780),(t>=780)&(t<960),t>=960],
        [lambda x:20*x/180,20,lambda x:20*(960-x)/180,0])

rng=np.random.default_rng(42)
t=np.arange(0,1101,1.0); z_true=true_depth(t)
bias_pa=350.0; noise_pa=250.0
p_meas=ambient_pressure_pa(z_true)+bias_pa+rng.normal(0,noise_pa,len(t))
z_meas=(p_meas-p0)/(rho*g)
plt.plot(t/60,z_true,label="true")
plt.plot(t/60,z_meas,alpha=.5,label="sensor-derived")
plt.gca().invert_yaxis(); plt.xlabel("Time [min]"); plt.ylabel("Depth [m]")
plt.title("Pressure noise and bias appear in derived depth"); plt.legend(); plt.show()

## 2. Differentiation amplifies noise; filtering adds lag

In [ ]:
def lowpass(x,alpha):
    y=np.empty_like(x); y[0]=x[0]
    for k in range(1,len(x)): y[k]=alpha*x[k]+(1-alpha)*y[k-1]
    return y

z_filt=lowpass(z_meas,0.08)
v_raw=np.gradient(z_meas,t)       # downward-positive derivative for display
v_filt=np.gradient(z_filt,t)
v_true=np.gradient(z_true,t)
fig,ax=plt.subplots(2,1,sharex=True,figsize=(8,7))
ax[0].plot(t/60,z_true,label="true"); ax[0].plot(t/60,z_filt,label="filtered")
ax[0].invert_yaxis(); ax[0].set_ylabel("Depth [m]"); ax[0].legend()
ax[1].plot(t/60,v_raw,alpha=.35,label="raw derivative")
ax[1].plot(t/60,v_filt,label="filtered derivative"); ax[1].plot(t/60,v_true,"k--",label="true")
ax[1].set(xlabel="Time [min]",ylabel="Descent rate [m/s]"); ax[1].legend(); plt.show()

## 3. Sampling-rate experiment

In [ ]:
for step in [1,5,15]:
    zz=z_meas[::step]; tt=t[::step]
    vv=np.gradient(zz,tt)
    print(f"sample every {step:2d} s: derivative noise SD {np.std(vv-np.gradient(z_true[::step],tt)):.3f} m/s")

## 4. Estimated gas state

In [ ]:
surface_rate=18.0
rate=surface_rate*ambient_pressure_pa(np.maximum(z_filt,0))/p0
used=np.concatenate([[0],np.cumsum((rate[1:]+rate[:-1])*np.diff(t/60)/2)])
g_est=2400-used
plt.plot(t/60,g_est)
plt.xlabel("Time [min]"); plt.ylabel("Estimated gas [surface L]")
plt.title("Model-propagated resource state"); plt.show()

## 5. Plausibility and threshold logic

In [ ]:
p_fault=p_meas.copy(); p_fault[500:520]=p_fault[499]; p_fault[800]+=200000
dp=np.diff(p_fault,prepend=p_fault[0])
stuck=np.r_[False,np.isclose(np.diff(p_fault),0,atol=1e-9)]
jump=np.abs(dp)>30000
print("Stuck samples detected:",np.count_nonzero(stuck))
print("Implausible jumps detected:",np.flatnonzero(jump))

## 6. Measurement classification

| Quantity | Status in this notebook |
|---|---|
| Pressure | measured (simulated sensor) |
| Depth | derived from pressure |
| Vertical speed | estimated by filtered differentiation |
| Remaining gas | propagated model state |
| Future remaining time | conditional prediction |

## Engineering exercises

1. Estimate and remove surface pressure bias from the initial surface samples.
2. Tune $\alpha$ and quantify noise–lag tradeoff.
3. Implement missing-sample detection.
4. Replace filtered differentiation with the Kalman filter from Notebook 08.

## Summary

A dive-computer display mixes measurements and models. Explicit classification, uncertainty, fault checks, and validation boundaries are essential parts of the engineering design.